# Statistics I: describing data honestly

Data Science & AI. Module 1 Part 2.

The deck opens with the right claim *(slide 41)*: **statistical thinking**
is the data-driven mindset. Its two load-bearing ideas, tonight and
forever:

- We almost never see the whole **population**; we see a **sample**, so
  every number we compute carries **uncertainty**.
- Sampling has to be **random** — a sample of volunteers, or of survivors,
  quietly measures the wrong thing.
- The gold standard for *measuring* anything is a **controlled
  experiment** — change one thing, hold the rest still. (You already
  live by this rule when debugging code.)

Tonight is the *describing* half of statistics: turning a table of raw
values into a handful of honest numbers and pictures. Wednesday is the
*inference* half — deciding what those numbers prove.

The deck also says the best way to learn this is *experimenting with data
in Python* — so, as always, everything runs.

## Before you type anything

**Work on your own copy, not on this file**, then **Restart & Run All**.

In [ ]:
# ==========================================================
# Setup. Run this once, then carry on.
# ==========================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RAW = "https://raw.githubusercontent.com/rejusam/data-science-ai-course/main/"
hr = pd.read_csv(RAW + "data/employee-attrition.csv")

rng = np.random.default_rng(0)
pd.set_option("display.max_columns", 20)
print("rows:", len(hr))

In [ ]:
hr

## 2. Categorical variables  *(slides 43, 44, 45)*

Two kinds of column live in every dataset, and they get different
statistics. **Categorical** columns hold labels — true/false, colour,
species, department *(slide 43)*. You cannot average a department.

What you *can* do is **count**. A **frequency table** *(slide 44)* is
category incidence in one variable — `value_counts`, which you know, plus
its cousin `normalize=True` for proportions:

In [ ]:
print(hr["STATUS"].value_counts())
print()
print(hr["STATUS"].value_counts(normalize=True).round(3))

The deck draws frequency tables as **donut charts** *(slide 44)* — a pie
with a hole. Fine for two or three categories; past that, bars beat
slices (angles are hard to compare by eye):

In [ ]:
counts = hr["STATUS"].value_counts()
# print(counts)
print(counts.index)
plt.figure(figsize=(4, 4))
plt.pie(counts, labels=counts.index,  autopct="%1.1f%%", wedgeprops={"width": 0.40})           # the hole makes it a donut
plt.title("The deck's donut chart, from our own data")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Optional: Create an explode array to slightly offset each wedge based on the length of 'counts'
explode_arr = [0.05] * len(counts) 

plt.pie(
    x=counts,                            # The wedge sizes
    explode=explode_arr,                 # Fraction of the radius to offset each wedge
    labels=counts.index,                 # Strings providing the labels for each wedge
    colors=None,                         # Sequence of colors (None uses current active cycle)
    autopct="%1.1f%%",                   # Format string used to label the wedges with their numeric value
    pctdistance=0.75,                    # Relative distance along the radius for the autopct text
    shadow=True,                         # Draw a shadow beneath the pie (can also take a dict for styling)
    labeldistance=1.1,                   # Relative distance along the radius for the labels
    startangle=90,                       # Angle by which the start of the pie is rotated (90 starts at the top)
    radius=1.0,                          # The radius of the pie
    counterclock=True,                   # Specify fractions direction (True = counterclockwise)
    wedgeprops={
        "width": 0.45,                   # Retained your donut chart width
        "edgecolor": "white",            # Added a white border between slices
        "linewidth": 1.5                 # Thickness of the borders
    }, 
    textprops={
        "fontsize": 10,                  # Control label and autopct font size
        "color": "#333333"               # Control label and autopct text color
    }, 
    center=(0, 0),                       # Coordinates of the center of the chart
    frame=False,                         # Plot Axes frame with the chart
    rotatelabels=False,                  # Rotate each label to the angle of the corresponding slice
    normalize=True,                      # Always make a full pie by normalizing x so sum(x) == 1
    hatch=None,                          # Hatching pattern (e.g., ['/', '\\', '|', '-']) (Requires Matplotlib 3.7+)
    data=None                            # Indexable object if passing string keys for x, explode, etc.
)

A **two-way table** *(slide 45)* counts one variable split by another —
in pandas, `pd.crosstab`. The totals along the edges are the **marginal
frequencies**, the deck's word because they live in the margins:

In [ ]:
two_way = pd.crosstab(hr["BUSINESS_UNIT"], hr["STATUS"], margins=True)
display(two_way)
two_way = pd.crosstab(hr["STATUS"], hr["BUSINESS_UNIT"], margins=True)
display(two_way)
two_way = pd.crosstab(hr["STATUS"], hr["BUSINESS_UNIT"], margins=False)
display(two_way)

Read one row aloud to check yourself: of the head-office rows, how many
are terminated, how many active, and do they sum to the margin? A
two-way table is the workhorse of every "does group A differ from
group B?" question — Wednesday's session will *test* such differences;
tonight we only tabulate them.

### Dummy variables  *(slides 46, 47)*

Models do arithmetic, and arithmetic needs numbers, so categorical
columns must become numeric *without inventing fake order*. Coding
STORE/HEADOFFICE as 1/2 would whisper "head office is twice a store".
**Dummy coding** *(slide 46)* avoids the lie: one 0/1 column per
category.

In [ ]:
hr

In [ ]:
# termreason_desc has FOUR categories -- watch what get_dummies does:
#print(hr["termreason_desc"].unique())

pd.get_dummies(hr["BUSINESS_UNIT"], dtype=int).head()

Exactly the deck's slide-47 table — four categories in, four 0/1
columns out, a single 1 per row marking which category it was. That is
the whole rule, at any size: **k categories become k columns**; nothing
about the recipe changes when k grows. File it away: `get_dummies`
(sklearn calls it one-hot encoding) reappears the moment Module 4 needs
a department inside a regression.

## 3. Continuous variables: where is the centre?  *(slides 49, 50, 51)*

**Continuous** columns hold measurements — height, dose, revenue, age
*(slide 49)*. Counting distinct values is meaningless now; the questions
become *where is the centre?* and *how spread out?*

First look is always the **histogram** *(slide 50)*: cut the range into
bins, count what falls in each — the shape of the column.

In [ ]:
plt.figure(figsize=(8, 3.5))
plt.hist(hr["age"], bins=30, edgecolor="white")
plt.xlabel("age"); plt.ylabel("employees")
plt.title("Distribution of age")
plt.grid(alpha=0.3)
plt.show()

Three centres, from slide 51 — you met them in `describe()`; tonight,
what they *mean* and when they disagree:

- **mean** — the average; balances the histogram like a see-saw
- **median** — the middle value; half below, half above
- **mode** — the most common value

In [ ]:
print("mean  :", hr["age"].mean().round(2))
print("median:", hr["age"].median())
print("mode  :", hr["age"].mode()[0])

> **Predict first.** Salaries at a company: eight people on $\$$60k, one on $\$$1.2m. Which is bigger, the mean or the median — and which one would the recruiter quote?
>
> Put your answer in the chat before we run it.

In [ ]:
salaries = pd.Series([60, 60, 60, 60, 60, 60, 60, 60, 1200])   # in $1000s
print("mean  :", round(salaries.mean(), 1), "thousand")
print("median:", salaries.median(), "thousand")

Mean $\$$186k, median $\$$60k. One executive dragged the mean past everyone in
the room — **the mean chases outliers, the median stands still**. Neither
is "right": the mean is the accountant's number (it times headcount gives
total payroll), the median is the honest "typical person" number. A
skewed distribution is precisely one where they disagree, and quoting the
flattering one is the oldest statistical sleight of hand there is.

## 4. Spread: quantiles, Interquartile range(IQR), standard deviation  *(slides 52, 53, 54, 56)*

Two datasets can share a mean and be nothing alike. Two emergency
departments both average a 60-minute wait — but one is a metronome and
the other is a lottery, and as a patient you care about the difference
more than the mean:

In [ ]:
metronome = rng.normal(60, 4, 5000)      # waits centred on 60, spread 4
lottery = rng.normal(60, 15, 5000)       # same centre 60, spread 15

plt.figure(figsize=(8, 3.5))
plt.hist(lottery, bins=50, alpha=0.6, label="ED B: sd 15", edgecolor="white")
plt.hist(metronome, bins=50, alpha=0.6, label="ED A: $\sigma$ 4", edgecolor="white")
plt.axvline(60, color="black", ls="--")  # the shared mean
plt.xlabel("wait (minutes)"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Identical means; only spread tells them apart")
plt.show()

print("ED A: mean", round(metronome.mean()), "  sd", round(metronome.std()))
print("ED B: mean", round(lottery.mean()), "  sd", round(lottery.std()))
print("share waiting over 90 minutes -- A:", (metronome > 90).mean(),
      "  B:", round((lottery > 90).mean(), 3))

Spread is the other half of any summary — and it is where the risk
lives: ED B's *mean* never warns you that one patient in fifty waits
past an hour and a half.

**Quantiles** *(slide 52)* run the histogram backwards: instead of "how
many in this bin?", ask "what value has 25% of the data below it?"
(Same idea per cent is a **percentile** *(slide 55)*: the 0.25 quantile
and the 25th percentile are one thing with two names.)

In [ ]:
hr[["EmployeeID", "length_of_service"]]

In [ ]:
print(hr["length_of_service"].quantile([0.25, 0.50, 0.75]))

The middle two — 0.25 to 0.75 — bracket the central half of the data;
their width is the **interquartile range (IQR)** *(slide 53)*. It is the
box in every box plot, the whiskers reach 1.5 × IQR further, and points
beyond the whiskers are the standard definition of **outliers**:

In [ ]:
plt.figure(figsize=(8, 2.4))
sns.boxplot(x=hr["length_of_service"], color="lightsteelblue")
plt.title("Box = IQR, line = median; dots past the whiskers would be outliers (none here)")
plt.show()

q1, q3 = hr["length_of_service"].quantile([0.25, 0.75])
iqr = q3 - q1
n_out = (hr["length_of_service"] > q3 + 1.5 * iqr).sum()
print("IQR = {:.0f} years; outliers above the whisker: {}".format(iqr, n_out))

The other spread measure is built from the mean ($\mu\: \text(aka) \bar{x}= \dfrac{1}{n} \sum_{i=1}^{n} x_i$). Take every deviation
from the mean, square it (so left and right do not cancel), average —
that is the **variance** ($\sigma^2 = \dfrac{1}{n-1} \sum_{i=1}^{n} (x_i - \bar{x})^2$); square-root it back to the original units —
the **standard deviation** ($\surd(\dfrac{1}{n-1} \sum_{i=1}^{n} (x_i - \bar{x})^2) $)*(slides 54, 56)*. Once by hand, once by
pandas, and they must agree:

In [ ]:
x = hr["age"]
deviations = x - x.mean()
variance_by_hand = (deviations ** 2).sum() / (len(x) - 1)

print("variance by hand :", round(variance_by_hand, 3))
print("variance, pandas :", round(x.var(), 3))
print("std = sqrt(var)  :", round(np.sqrt(variance_by_hand), 3))
print("std, pandas      :", round(x.std(), 3))
# display(x)
# print(len(x))

Two footnotes the deck flags and interviews love:

- The divisor is **n − 1**, not n, for a *sample* — the tiny correction
  for the fact that we measured the mean from the same data. Wednesday's
  session says more.
- Slide 54's other two **moments** describe shape: **skewness**
  (asymmetry — the salary example was heavily right-skewed) ($\frac{1}{n-1} \sum_{i=1}^{n} (x_i - \bar{x})^3$) and
  **kurtosis** ($\frac{1}{n-1} \sum_{i=1}^{n} (x_i - \bar{x})^4 - 3$) -- how heavy the tails are. `hr["age"].skew()` when you
  need them; no more tonight.

In [ ]:
print(hr["age"].skew())
print(hr["age"].kurtosis())

In [ ]:
salaries = pd.Series([60, 60, 60, 60, 60, 60, 60, 60, 1200])
salaries.skew()

## 5. The z-score: one ruler for everything  *(slides 57, 58, 59, 60)*

Is a 55-year-old unusual in this company? Is 24 years of service? The
questions live on different scales, so raw values cannot answer "which is
*more* unusual". The fix *(slide 60)*: measure everything in standard
deviations from its own mean.

**z = (x − mean) / std** — "how many standard deviations from typical".

It is a unit conversion, nothing more: instead of metres or years or
dollars, measure every quantity in **its own natural unit of spread** —
the way a hiker states distance in "days' walk" so that a desert and a
mountain range become comparable. One spread = ordinary, two = notable,
three = rare, whatever the raw units were.

In [ ]:
def z_score(series, value):
    return (value - series.mean()) / series.std()

print("age 55              -> z = {:.2f}".format(z_score(hr["age"], 42.5)))
print("service 24 years    -> z = {:.2f}".format(z_score(hr["length_of_service"], 30)))

Same ruler, so now they compare — the long-server is the more unusual.
(Slide 57's notation, for reading the literature: sample statistics wear
Latin letters, x̄ and s; population parameters wear Greek, μ and σ. Same
recipes, different data.)

Why z-scores matter so much: many real measurements pile up in the famous
bell shape — the **normal distribution** *(slides 58, 59)* — and on that
curve, z *is* an address. The rule worth memorising:

| within | share of data |
|---|---|
| ± 1 σ | ~68% |
| ± 2 σ | ~95% |
| ± 3 σ | ~99.7% |

Check the claim against ten thousand simulated systolic blood pressures:

In [ ]:
bp = rng.normal(120, 15, 10_000)          # mean 120, sd 15

z = (bp - bp.mean()) / bp.std()
for k in [1, 2, 3]:
    share = (np.abs(z) < k).mean()
    print("within ±{} sd: {:.1%}".format(k, share))

plt.figure(figsize=(8, 3.5))
plt.hist(bp, bins=60, alpha=0.75, edgecolor="white")
for k in [1, 2, 3]:
    plt.axvline(120 - k * 15, color="grey", ls="--")   # fences at 1, 2, 3 sd
    plt.axvline(120 + k * 15, color="grey", ls="--")
plt.title("Simulated blood pressure: 68 / 95 / 99.7 inside the fences")
plt.xlabel("systolic BP"); plt.grid(alpha=0.3)
plt.show()

68 / 95 / 99.7, as promised. This is also your first **outlier detector
with a principle**: on roughly-normal data, |z| > 3 happens 3 times in a
thousand by chance — a z of 6 is not bad luck, it is a data-entry error
or a discovery. And "z = 2 is the edge of the usual 95%" is the exact
fact Wednesday's p-values are built on. Same curve, same fences.

(One caution before trusting z everywhere: the salary example was not
normal — heavily skewed data breaks the 68/95/99.7 promise. Look at the
histogram first; that is what it is for.)

## 6. Two variables at once  *(slides 62, 63, 64, 65)*

Everything so far described columns alone. The interesting questions
join two: does service length move with age? The first tool is never a
number — it is the **scatter plot** *(slide 62)*:

In [ ]:
sample = hr.sample(400, random_state=0)
plt.figure(figsize=(7, 4))
plt.scatter(sample["age"], sample["length_of_service"], s=14, alpha=0.5)
plt.xlabel("age"); plt.ylabel("length of service (years)")
plt.title("400 employees")
plt.grid(alpha=0.3)
plt.show()

To put one number on "how much do they move together", the **Pearson
correlation coefficient r** *(slide 63)*: multiply each point's
z-score-in-x by its z-score-in-y and average. Points that are high in
both or low in both push r up; mismatches push it down.

- r ≈ +1 — tight uphill line
- r ≈ −1 — tight downhill line
- r ≈ 0 — no *linear* relationship

In [ ]:
print("r, by the every-pair method:")
print(hr[["age", "length_of_service", "STATUS_YEAR"]].corr().round(3))

`.corr()` computes r for every pair at once — the **correlation matrix**,
ones down the diagonal because everything correlates perfectly with
itself. Age and service: r ≈ 0.91 — a *strong*, tight uphill
relationship, and the scatter shows why: nobody's service can exceed
their age, and much of this workforce joined young and stayed, so the
two columns nearly move in lockstep. STATUS_YEAR against age, by
contrast: r ≈ −0.04 — knowing the record's year tells you essentially
nothing about the employee's age.

> **Predict first.** Slide 64 shows four datasets that all share r = 0.816. Before we draw them: can four scatter plots with the same correlation look genuinely different, or must they roughly agree?
>
> Put your answer in the chat before we run it.

In [ ]:
ans = sns.load_dataset("anscombe")

sns.lmplot(data=ans, x="x", y="y", col="dataset", ci=None, height=2.6)
plt.show()

for name, part in ans.groupby("dataset"):
    print("dataset", name, ":  r =", round(part["x"].corr(part["y"]), 3))

**Anscombe's quartet** *(slide 64)*: a clean line, a curve, an outlier
wrecking a perfect fit, and a vertical stack with one point doing all the
work — identical r throughout. The deck's warning, now in your own
output: *r assumes a straight line and does not fully characterise a
relationship.* The rule it buys you: **never report a correlation you
have not plotted.** Thirty years of dashboards have been wrong for
skipping that sentence.

## 7. The trend line  *(slides 65, 66, 67)*

The scatter suggests a line; which line? For each candidate, measure
every point's vertical miss — the **residual** εᵢ = yᵢ − ŷᵢ *(slide 66)*
— square the misses, add them up. **Least-squares regression** picks the
line with the smallest total *(slide 67)*.


In [ ]:
demo = hr.sample(25, random_state=1)
x25 = demo["age"].to_numpy()
y25 = demo["length_of_service"].to_numpy()
m25, c25 = np.polyfit(x25, y25, 1)

plt.figure(figsize=(7.5, 4.5))
plt.scatter(x25, y25, s=40, zorder=5)
grid_x = np.array([x25.min() - 2, x25.max() + 2])
plt.plot(grid_x, m25 * grid_x + c25, color="tab:red", lw=2,
         label="the settled rod (least-squares line)")
for xi, yi in zip(x25, y25):
    plt.plot([xi, xi], [yi, m25 * xi + c25], color="tab:green",
             lw=1.5, alpha=0.8)
plt.xlabel("age"); plt.ylabel("length of service")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

residuals = y25 - (m25 * x25 + c25)
print("squared residuals:", round((residuals ** 2).sum(), 1))
print("residuals sum:", residuals.sum())

You built the same minimisation on Wednesday from the calculus side —
gradient descent walking the loss downhill. The deck's slide-67 formulas
are the closed-form shortcut, and `np.polyfit` runs them on the full
dataset:

In [ ]:
slope, intercept = np.polyfit(hr["age"], hr["length_of_service"], 1)
print("service ≈ {:.3f} × age {:+.2f}".format(slope, intercept))

plt.figure(figsize=(7, 4))
plt.scatter(sample["age"], sample["length_of_service"], s=15, alpha=0.4)
ages = np.array([18, 65])
plt.plot(ages, slope * ages + intercept, color="tab:red", lw=2,
         label="least-squares line")
plt.xlabel("age"); plt.ylabel("length of service")
plt.legend(); plt.grid(alpha=0.3)
plt.title("The line that minimises the squared misses")
plt.show()

Read the slope like a data scientist: *each extra year of age goes with
almost half a year more service, on average.* Two honesty clauses
attach to every such sentence:

- **"goes with", not "causes"** — older employees also joined in
  different eras; correlation orders none of that.
- The line summarises **this sample's range**. At age 90 it predicts
  confidently; we have no 90-year-olds. Extrapolation is fiction with
  good posture.

Why it is called "regression" at all: Francis Galton, in the 1880s,
fitted exactly this kind of line to parents' and children's heights and
found children of very tall parents were tall, but *less* tall —
pulled back toward the average. He called it "regression toward
mediocrity"; the phenomenon is real (it is what r < 1 *means*:
an extreme in x predicts a milder extreme in y), the gloomy name stuck
to the method, and statisticians have been explaining the word ever
since. The trap named after it: sports stars slump after a great
season, patients improve after their worst week — mostly not cause and
effect, just extremes drifting back toward the middle on the next draw.

Module 4 picks up exactly here — same line, plus the machinery for *how
good is the fit* (R², the fourth Anscombe panel's downfall) and *what
can we claim* (Wednesday's inference).

---

## Your turn

The official lab is IOD Lab 1.2.3 (Statistics). Warm-ups first — all on
the attrition table.

In [ ]:
# 1. Frequency table of "gender_full", as counts and as proportions.

In [ ]:
# 2. Two-way table: department_name (rows) by STATUS (columns), with
#    margins. Which department has the most terminations in raw count?

In [ ]:
# 3. For "STATUS_YEAR": mean, median, IQR. One comment: is it skewed?

In [ ]:
# 4. z-score of a 64-year-old in this company, saved as `z64`.
#    Roughly what share of a normal curve lies beyond that z?

In [ ]:
# 5. Correlation matrix of the numeric columns of `hr`. Which pair
#    correlates most strongly (excluding the diagonal)? Scatter-plot
#    that pair on a 400-row sample before you believe the number.

### Stretch

1. Draw the age histogram for ACTIVE and TERMINATED on the same axes
   (`alpha=0.5`, `density=True`). Where do the shapes differ? Keep the
   picture in mind — Wednesday we *test* whether that difference is real.
2. `hr["length_of_service"].skew()` — sign and size, and what the
   histogram says about why.
3. Anscombe by hand: for dataset "II" compute mean of x, mean of y, and
   r yourself with the z-score-product recipe, and match `.corr()`.

### If you want the pictures again, slower

The see-it-first style of the last two sessions (Dr Trefor Bazett,
3Blue1Brown) has statistics counterparts, all free:

- **StatQuest with Josh Starmer** —
  [youtube.com/@statquest](https://www.youtube.com/@statquest): short,
  rigorous, drawn-by-hand videos on every idea tonight — mean/variance,
  the normal distribution, correlation, least squares — and later on
  most of Modules 4 and 5.
- **Seeing Theory** (Brown University) —
  [seeing-theory.brown.edu](https://seeing-theory.brown.edu): interactive
  visualisations you drive yourself; the *Basic Probability* and
  *Regression Analysis* chapters map onto tonight's sections.
- **Khan Academy** —
  [khanacademy.org/math/statistics-probability](https://www.khanacademy.org/math/statistics-probability):
  worked exercises with instant feedback — the deck's statistics section
  follows this curriculum's order almost exactly.

---

*Data Science & AI — Session 8. Covers Module 1 Part 2 slides 40–67.
Official lab: IOD Lab 1.2.3 Statistics (`labs/lab1/`).*